# 02 - TF-IDF One-vs-Rest Logistic Regression & Threshold Tuning

In [2]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import joblib
import json
import os
import sys
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.multiclass import OneVsRestClassifier

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../"))
if PROJECT_ROOT not in sys.path: 
    sys.path.insert(0, str(PROJECT_ROOT))
from manual_ovr_logistic_regression import ManualLogisticRegression


## 1. Load TF-IDF Features

In [3]:
aspect_cols = ['food', 'service', 'price', 'ambiance', 'miscellaneous']

X_train_tfidf = sp.load_npz(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "X_train_tfidf.npz"))
X_val_tfidf = sp.load_npz(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "X_val_tfidf.npz"))

y_train = pd.read_csv(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "y_train.csv"))
y_val = pd.read_csv(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "y_val.csv"))


## 2. Grid Search for OVR Logistic Regression

In [4]:
# Define base model
base_lr = ManualLogisticRegression(random_state=42, max_iter=1000)
classifier = OneVsRestClassifier(base_lr)

# Define parameter grid
param_grid = {
    'estimator__C': [0.1, 1, 10],
    'estimator__class_weight': [None, 'balanced'],
    'estimator__lr': [0.01, 0.001]
}

# Run Grid Search
print("Running Grid Search...")
grid_search = GridSearchCV(
    classifier, 
    param_grid, 
    scoring='f1_macro', 
    cv=3, 
    verbose=0, 
    n_jobs=-1
)
grid_search.fit(X_train_tfidf, y_train.values)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
print("Best parameters:", best_params)


Running Grid Search...
Best parameters: {'estimator__C': 0.1, 'estimator__class_weight': 'balanced', 'estimator__lr': 0.01}


## 3. Tune Thresholds on Validation Set

In [5]:
y_val_prob = best_model.predict_proba(X_val_tfidf)

def tune_thresholds(y_true, y_prob, aspect_cols):
    thresholds = {}
    best_f1_scores = {}
    
    for i, col in enumerate(aspect_cols):
        best_f1 = 0
        best_thresh = 0.5
        for thresh in np.arange(0.1, 0.9, 0.05):
            pred = (y_prob[:, i] >= thresh).astype(int)
            f1 = f1_score(y_true.iloc[:, i], pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = thresh
                
        thresholds[col] = float(np.round(best_thresh, 2))
        best_f1_scores[col] = float(np.round(best_f1, 4))
        print(f"Optimal threshold for {col}: {thresholds[col]:.2f} (Val F1: {best_f1_scores[col]:.4f})")
        
    return thresholds, best_f1_scores

optimal_thresholds, val_f1_scores = tune_thresholds(y_val, y_val_prob, aspect_cols)


Optimal threshold for food: 0.10 (Val F1: 0.9184)
Optimal threshold for service: 0.50 (Val F1: 0.8315)
Optimal threshold for price: 0.50 (Val F1: 0.9388)
Optimal threshold for ambiance: 0.50 (Val F1: 0.8211)
Optimal threshold for miscellaneous: 0.50 (Val F1: 0.3750)


## 4. Save Classifier and Metadata

In [6]:
classifiers_dir = os.path.join(PROJECT_ROOT, "models", "TF-IDF", "classifiers")
os.makedirs(classifiers_dir, exist_ok=True)

# Save model
model_path = os.path.join(classifiers_dir, "tfidf_ovr_logreg.joblib")
joblib.dump(best_model, model_path)

# Save metadata
metadata = {
    'best_params': best_params,
    'thresholds': optimal_thresholds,
    'validation_f1_scores': val_f1_scores
}

with open(os.path.join(classifiers_dir, "tfidf_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

print("Classifier and metadata saved successfully!")

Classifier and metadata saved successfully!
